# Customer Migration to OneBill — TEST

This notebook fetches **Consumer contacts** from Microsoft Dataverse and prepares them for migration into OneBill.

**Steps:**
1. Authenticate with Dataverse via OAuth2
2. Fetch all Consumer contacts using FetchXML (with full pagination)
3. Expand multi-value contact types into individual rows
4. Output a clean DataFrame ready for the OneBill migration pipeline

## Imports

Install `msal` if not already present, then import all required libraries.

In [65]:
# %pip install msal
import os
import re
import urllib.parse
import urllib.parse
from html import escape as xml_escape
import requests
from msal import ConfidentialClientApplication
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 11
python-dotenv could not parse statement starting at line 14


True

## Configuration

Credentials and environment URL are loaded from a `.env` file. Ensure the following variables are set:

| Variable | Description |
|---|---|
| `CRM_TENANT_ID` | Azure AD tenant ID |
| `CRM_CLIENT_ID` | App registration client ID |
| `CRM_CLIENT_SECRET` | App registration client secret |
| `CRM_ENVIRONMENT_URL` | Dataverse environment URL (no trailing slash), e.g. `https://yourorg.crm.dynamics.com` |

In [2]:
TENANT_ID       = os.environ["CRM_TENANT_ID"]
CLIENT_ID       = os.environ["CRM_CLIENT_ID"]
CLIENT_SECRET   = os.environ["CRM_CLIENT_SECRET"]
ENVIRONMENT_URL = os.environ["CRM_ENVIRONMENT_URL"]

## FetchXML Query

Defines the Dataverse query that retrieves all active contacts whose parent account has `accountcategorycode = 1` (Consumer segment).

The `link-entity` performs an inner join to the `account` table, ensuring only contacts belonging to a Consumer account are returned. The `accountnumber` field is included from the account so it is available for the OneBill migration.

In [74]:
FETCHXML = """
<fetch version="1.0" output-format="xml-platform" mapping="logical" no-lock="false">
  <entity name="contact">
    <attribute name="statecode"/>
    <attribute name="entityimage_url"/>
    <attribute name="fullname"/>
    <attribute name="emailaddress1"/>
    <attribute name="telephone1"/>
    <attribute name="contactid"/>
    <attribute name="vgr_contacttypes"/>
    <attribute name="mobilephone"/>
    <attribute name="firstname"/>
    <attribute name="lastname"/>
    <order attribute="contactid" descending="false"/>
    <filter type="and">
      <condition attribute="statecode" operator="eq" value="0"/>
    </filter>
    <link-entity name="account" from="accountid" to="parentcustomerid" link-type="inner" alias="a_ce3f647c660e4a5c99cc3d631f23406d">
      <attribute name="accountnumber"/>
      <filter type="and">
        <condition attribute="accountcategorycode" operator="eq" value="1"/>
        <condition attribute="accountnumber" operator="not-null"/>
      </filter>
    </link-entity>
  </entity>
</fetch>
"""

## Authentication

Obtains an OAuth2 bearer token from Azure AD using the client credentials flow.
The token is scoped to the Dataverse environment and is valid for ~1 hour.

In [75]:
def get_access_token() -> str:
    """Acquire an OAuth2 bearer token for Dataverse via client credentials flow."""
    app = ConfidentialClientApplication(
        client_id=CLIENT_ID,
        client_credential=CLIENT_SECRET,
        authority=f"https://login.microsoftonline.com/{TENANT_ID}",
    )
    result = app.acquire_token_for_client(scopes=[f"{ENVIRONMENT_URL}/.default"])
    if "access_token" not in result:
        raise RuntimeError(f"Token acquisition failed: {result.get('error_description')}")
    return result["access_token"]

## Fetch Contacts from Dataverse

Executes the FetchXML query against the Dataverse Web API and handles **pagination** automatically.

Dataverse returns a maximum of 5,000 records per page. When more records exist, the response includes:
- `@Microsoft.Dynamics.CRM.morerecords: true` — signals another page is available
- `@Microsoft.Dynamics.CRM.fetchxmlpagingcookie` — an XML cookie that must be injected back into the FetchXML for the next page request

The paging cookie is XML-escaped (not URL-encoded) before being embedded into the FetchXML `page` attribute.

In [78]:
def get_contacts(token: str) -> pd.DataFrame:
    headers = {
        "Authorization": f"Bearer {token}",
        "OData-MaxVersion": "4.0",
        "OData-Version": "4.0",
        "Accept": "application/json",
        "Prefer": 'odata.maxpagesize=5000',
    }

    all_records = []
    last_contactid = None
    page = 1

    while True:
        # Build FetchXML with a "contactid > last_seen" filter
        if last_contactid is None:
            extra_condition = ""
        else:
            extra_condition = f'<condition attribute="contactid" operator="gt" value="{last_contactid}"/>'

        fetch = f"""
<fetch version="1.0" output-format="xml-platform" mapping="logical" no-lock="false" count="5000">
  <entity name="contact">
    <attribute name="statecode"/>
    <attribute name="entityimage_url"/>
    <attribute name="fullname"/>
    <attribute name="emailaddress1"/>
    <attribute name="telephone1"/>
    <attribute name="contactid"/>
    <attribute name="vgr_contacttypes"/>
    <attribute name="mobilephone"/>
    <attribute name="firstname"/>
    <attribute name="lastname"/>
    <order attribute="contactid" descending="false"/>
    <filter type="and">
      <condition attribute="statecode" operator="eq" value="0"/>
      {extra_condition}
    </filter>
    <link-entity name="account" from="accountid" to="parentcustomerid" link-type="inner" alias="a_ce3f647c660e4a5c99cc3d631f23406d">
      <attribute name="accountnumber"/>
      <filter type="and">
        <condition attribute="accountcategorycode" operator="eq" value="1"/>
        <condition attribute="accountnumber" operator="not-null"/>
      </filter>
    </link-entity>
  </entity>
</fetch>
"""
        url = f"{ENVIRONMENT_URL}/api/data/v9.2/contacts?fetchXml={urllib.parse.quote(fetch)}"
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()

        records = data.get("value", [])
        if not records:
            break

        all_records.extend(records)
        new_last = records[-1]["contactid"]
        print(f"Page {page}: fetched {len(records):,} records "
              f"(last contactid={new_last}, total so far: {len(all_records):,})")

        # If we got fewer than the page size, we're done
        if len(records) < 5000:
            break

        # Safety guard: if the last contactid hasn't advanced, stop
        if new_last == last_contactid:
            print("contactid did not advance — stopping")
            break

        last_contactid = new_last
        page += 1

        if page > 50:
            print("Hit MAX_PAGES safety limit")
            break

    df = pd.DataFrame(all_records)
    df.rename(columns=lambda c: c.replace("aa.", "account_"), inplace=True)
    print(f"\nDone — {len(df):,} contacts loaded")
    return df

## Run — Fetch Contacts

Authenticate and pull all Consumer contacts from Dataverse into a DataFrame.

In [79]:
token = get_access_token()
df = get_contacts(token)
print(df.head())

Page 1: fetched 5,000 records (last contactid=4c0e43f0-e8e1-e911-a812-000d3a79724a, total so far: 5,000)
Page 2: fetched 5,000 records (last contactid=91dc6244-0fa4-e911-a857-000d3ae02607, total so far: 10,000)
Page 3: fetched 5,000 records (last contactid=43afa486-0fa4-e911-a857-000d3ae02607, total so far: 15,000)
Page 4: fetched 5,000 records (last contactid=8036d6a4-0fa4-e911-a857-000d3ae02607, total so far: 20,000)
Page 5: fetched 5,000 records (last contactid=5071fec0-0fa4-e911-a857-000d3ae02607, total so far: 25,000)
Page 6: fetched 5,000 records (last contactid=b65f8fdd-0fa4-e911-a857-000d3ae02607, total so far: 30,000)
Page 7: fetched 5,000 records (last contactid=3da5431f-10a4-e911-a857-000d3ae02607, total so far: 35,000)
Page 8: fetched 5,000 records (last contactid=8e070698-c727-ec11-b6e6-0022481835fa, total so far: 40,000)
Page 9: fetched 5,000 records (last contactid=a4edbded-43ad-ec11-9840-002248d39307, total so far: 45,000)
Page 10: fetched 2,339 records (last contactid=

## Contact Type Configuration

Dataverse stores contact types as multi-select numeric option set values (e.g. `287,790,000`). 
Dataverse formats these with thousands-separator commas, so they are parsed using a regex pattern rather than a simple split.

**Billing** and **Primary** are treated as communication preferences — they are combined into a single row rather than becoming individual OneBill contact type rows.

In [81]:
# Maps Dataverse numeric option set values to human-readable labels
CONTACT_TYPE_MAP = {
    "287790000": "Billing",
    "287790001": "Technical",
    "287790002": "Outage - Email",
    "287790009": "Outage - SMS",
    "287790003": "Primary",
    "287790004": "Technical - Data",
    "287790005": "Technical - Voice",
    "287790006": "Commercial",
    "287790008": "Communication",
    "287790007": "Voyager Staff",
}

# These two types are not individual OneBill contact types — they are combined
# as communication preferences and carried on every row for that contact
COMMUNICATION_PREFS = {"Billing", "Primary"}

## Expand Contact Types

Transforms each contact into multiple rows — one per OneBill contact type — according to these rules:

- If a contact has **Billing** and/or **Primary**, these are combined into a single row where `contact_type` is set to `"Billing; Primary"` (or whichever apply), and `is_billing` / `is_primary` are `True`
- Every other contact type (Technical, Commercial, etc.) becomes its own row, with `is_billing` and `is_primary` set to `False`
- If a contact has no types at all, a single row is kept with `contact_type = None`

**Example:**

| Input | Rows out | contact_type | is_billing | is_primary |
|---|---|---|---|---|
| `Billing; Primary; Technical` | 2 | `Billing; Primary` / `Technical` | `True` / `False` | `True` / `False` |
| `Billing; Technical; Commercial` | 3 | `Billing` / `Technical` / `Commercial` | `True` / `False` / `False` | `False` / `False` / `False` |

In [82]:
def expand_contacts(df: pd.DataFrame) -> pd.DataFrame:
    """
    Splits vgr_contacttypes into individual rows per OneBill contact type.
    Billing and Primary are collapsed into a communication preferences row
    and carried through on every expanded row as boolean flags.
    """
    expanded_rows = []

    for _, row in df.iterrows():
        raw = str(row.get("vgr_contacttypes", "") or "")

        # Dataverse returns option set values formatted with thousands-separator commas
        # e.g. "287,790,000,287,790,003" — use regex to extract all 9-digit values
        raw_clean = raw.replace(",", "")
        type_values = re.findall(r"28779\d{4}", raw_clean)
        type_labels = [CONTACT_TYPE_MAP.get(v, v) for v in type_values]

        is_billing = "Billing" in type_labels
        is_primary = "Primary" in type_labels
        onebill_types = [t for t in type_labels if t not in COMMUNICATION_PREFS]

        base = {
            **row.to_dict(),
            "is_billing": is_billing,
            "is_primary": is_primary,
        }

        # Row 1 — communication preferences (Billing and/or Primary combined)
        if is_billing or is_primary:
            comm_prefs = "; ".join(t for t in ["Billing", "Primary"] if t in type_labels)
            expanded_rows.append({**base, "contact_type": comm_prefs})

        # Additional rows — one per OneBill contact type with flags set to False
        for contact_type in onebill_types:
            expanded_rows.append({
                **base,
                "contact_type": contact_type,
                "is_billing": False,
                "is_primary": False,
            })

        # If no types at all, keep the contact with a single placeholder row
        if not is_billing and not is_primary and not onebill_types:
            expanded_rows.append({**base, "contact_type": None})

    result = pd.DataFrame(expanded_rows)
    result.drop(columns=["vgr_contacttypes"], inplace=True, errors="ignore")
    result.reset_index(drop=True, inplace=True)

    return result

## Run — Expand Contact Types

Apply the expansion to the fetched contacts DataFrame and preview the result.

In [83]:
df_expanded = expand_contacts(df)

print(f"Original rows:  {len(df):,}")
print(f"Expanded rows:  {len(df_expanded):,}")
print(df_expanded[["fullname", "contact_type", "is_billing", "is_primary"]].head(20))

Original rows:  47,339
Expanded rows:  47,359
                 fullname contact_type  is_billing  is_primary
0            Garry Martin      Billing        True       False
1           Justin Bagust      Billing        True       False
2            Shanaya Keil      Billing        True       False
3          Michelle Child      Billing        True       False
4   Jonathan Clark-Howard      Billing        True       False
5           Hunter McLeod      Billing        True       False
6            Lennart Nout      Billing        True       False
7                Alf Mohi      Billing        True       False
8             Gene Hanham      Billing        True       False
9              Judy Sibbe         None       False       False
10         Tracker Apiata      Billing        True       False
11        Samantha Hilton      Billing        True       False
12          Loketi Fangia      Billing        True       False
13            Rob Sanders      Billing        True       False
14       

In [85]:
df_expanded['contact_type'].unique()

array(['Billing', None, 'Billing; Primary', 'Commercial', 'Primary',
       'Technical', 'Outage - Email', 'Outage - SMS', 'Voyager Staff',
       'Communication', 'Technical - Voice', 'Technical - Data'],
      dtype=object)